# Gigi × Colab：环境探针

这个 notebook 不做任何渲染工作，只回答一组**只有在真实 Colab 里才能回答**的问题，
它们决定 Gigi 的 Colab 版本该怎么设计：

| 探针 | 回答的问题 |
|---|---|
| kernel | VM 里有什么：Python、glibc、Node、Vulkan 驱动、GPU |
| frame | 输出 iframe 里 WebGPU 能不能用，用的是哪块显卡 |
| sibling_frames | 一个单元格的输出能否访问另一个单元格的输出 |
| comm | 浏览器 → 内核的异步通道（comm、invokeFunction）及其二进制载荷 |
| eval_js | 单元格执行期间的同步往返：延迟、两个方向各能传多大 |
| kernel_http | 内核里的后台 HTTP 服务能否在单元格执行期间被浏览器访问 |
| node_dawn | 内核里能否用 Node + Dawn 无头运行 WebGPU（Gigi 自己的测试路径） |
| persistence | 保存并重新打开后，输出里的 WebGPU 画面是否还活着（需要人工看一眼） |

**用法**：代码执行程序 → 全部运行（约 1–2 分钟），然后把最后一个单元格打印的 JSON 贴回来。
报告里包含浏览器 User-Agent 和显卡型号；域名里的会话 ID 已被替换成 `<id>`。

`eval_js` 探针会发送最大 16 MB 的消息来找上限。如果连接因此断开，这本身就是结果：
重新连接后运行最后一个单元格即可，已完成的部分保存在 VM 的 `/content/gigi_colab_probe.json`。

In [ ]:
import glob, json, os, platform, shutil, subprocess, sys, time
from importlib import metadata

REPORT = {"probe_version": 1}
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPORT_PATH = os.path.join(WORKDIR, "gigi_colab_probe.json")


def save():
    with open(REPORT_PATH, "w") as f:
        json.dump(REPORT, f, indent=1, default=str)


def sh(cmd, timeout=30, limit=600):
    """Run a shell command; return the tail of its output, or the error."""
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return (p.stdout + p.stderr).strip()[-limit:]
    except Exception as e:
        return f"<{type(e).__name__}: {e}>"


def version(name):
    try:
        return metadata.version(name)
    except Exception:
        return None


def probe(section, fn):
    """Run one probe, store its result (or its error) in REPORT, and print it."""
    t0 = time.time()
    try:
        REPORT[section] = fn()
    except Exception as e:
        REPORT[section] = {"error": f"{type(e).__name__}: {e}"}
    REPORT[section + "_seconds"] = round(time.time() - t0, 2)
    save()
    print(json.dumps(REPORT[section], indent=1, default=str)[:4000])


def kernel_facts():
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "libc": " ".join(platform.libc_ver()),
        "os": sh("grep PRETTY_NAME /etc/os-release"),
        "cpus": os.cpu_count(),
        "mem_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
        "disk_free_gb": round(shutil.disk_usage(WORKDIR).free / 2**30, 1),
        "colab_release": os.environ.get("COLAB_RELEASE_TAG"),
        "gpu": sh("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader"),
        "node": sh("node --version"),
        "npm": sh("npm --version"),
        "gcc": sh("gcc --version | head -1"),
        "cmake": sh("cmake --version | head -1"),
        "graphviz": sh("dot -V"),
        "vulkan_icd_files": sorted(glob.glob("/usr/share/vulkan/icd.d/*") + glob.glob("/etc/vulkan/icd.d/*")),
        "vulkan_libs": sh("ldconfig -p | grep -i -E 'libvulkan|libEGL' | head -6"),
        "packages": {n: version(n) for n in [
            "ipykernel", "ipython", "ipywidgets", "anywidget", "tornado",
            "jupyter-server", "numpy", "pillow", "jsonschema", "nbformat"]},
    }


probe("kernel", kernel_facts)

## 两个"先放着"的输出

下面两个单元格各自创建一个输出 iframe，后面的探针要用到它们：

- **frame A**：一个带标记的输出，稍后从*另一个*单元格尝试访问它。
- **comm 探针**：在浏览器里发起 comm / invokeFunction。这两条通道走内核的 shell 通道，
  内核只在**两次单元格执行之间**处理它们，所以结果要等后面的单元格跑起来才会陆续到达。

In [ ]:
from IPython.display import HTML, display

FRAME_A_HTML = r'''
<div id="gigi-probe-a" style="font: 13px monospace">frame A: starting…</div>
<script>
  // "Frame A": a marker that a later cell will try to reach.
  window.gigiProbeMarker = 'frame-A';
  window.gigiProbePing = (x) => 'pong:' + x;
  const gigiProbeLabel = document.getElementById('gigi-probe-a');
  try {
    const bc = new BroadcastChannel('gigi-probe');
    bc.onmessage = (ev) => {
      if (ev.data && ev.data.ping) {
        bc.postMessage({ pong: ev.data.ping });
        gigiProbeLabel.textContent = 'frame A: answered a broadcast from another cell';
      }
    };
  } catch (e) { /* no BroadcastChannel */ }
  const gigiProbeAccess = ((self.google || {}).colab || {}).kernel
    ? google.colab.kernel.accessAllowed : 'no google.colab';
  gigiProbeLabel.textContent = 'frame A: ready (kernel access: ' + gigiProbeAccess + ')';
</script>
'''

display(HTML(FRAME_A_HTML))

In [ ]:
import IPython
from google.colab import output

COMM_HTML = r'''
<div id="gigi-probe-comm" style="font: 13px monospace">comm probe: running…</div>
<script>
// The asynchronous routes from an output frame to the kernel:
//   invokeFunction - calls a Python function registered with output.register_callback
//   comms          - Jupyter comm channel, with binary buffers in both directions
// Both travel on the kernel's shell channel, so the kernel only handles them
// BETWEEN cell executions. That is why this probe reports through a comm message
// instead of being awaited by Python.
(async () => {
  const label = document.getElementById('gigi-probe-comm');
  const res = {};
  const kernel = ((self.google || {}).colab || {}).kernel;
  if (!kernel || !kernel.accessAllowed) {
    label.textContent = 'comm probe: no kernel access here (this is a saved output)';
    return;
  }
  const withTimeout = (p, ms) => Promise.race([p, new Promise((r) => setTimeout(() => r('<timeout>'), ms))]);
  try {
    const t0 = performance.now();
    const r = await withTimeout(kernel.invokeFunction('gigi_probe_callback', [1, 'two'], { three: 3 }), 120000);
    res.invoke = { ms: Math.round(performance.now() - t0), result: JSON.stringify(r).slice(0, 400) };
  } catch (e) { res.invokeError = String(e); }
  try {
    const comm = await kernel.comms.open('gigi_probe_comm', { hello: 1 }, [new Uint8Array(1000).buffer]);
    const incoming = comm.messages[Symbol.asyncIterator]();
    res.comm = [];
    for (const n of [100000, 1000000, 8000000]) {
      const t0 = performance.now();
      await comm.send({ echo_bytes: n }, { buffers: [new Uint8Array(n).buffer] });
      const m = await withTimeout(incoming.next(), 120000);
      if (m === '<timeout>' || m.done) { res.comm.push({ bytes: n, ok: false, why: String(m.done ? 'closed' : m) }); break; }
      const got = (m.value.buffers || []).reduce((a, b) => a + b.byteLength, 0);
      res.comm.push({ bytes: n, ok: got === n, roundtripMs: Math.round(performance.now() - t0) });
    }
    await comm.send({ final: res });
    comm.close();
  } catch (e) { res.commError = String(e); }
  label.textContent = 'comm probe: ' + JSON.stringify(res);
})();
</script>
'''

COMM_LOG = []  # what the kernel received on the comm


def _on_comm_open(comm, open_msg):
    COMM_LOG.append({"event": "open", "buffers": [len(b) for b in open_msg.get("buffers", [])]})

    def on_msg(msg):
        data = msg["content"].get("data") or {}
        if "final" in data:  # the browser's own summary of the whole exchange
            REPORT["comm"] = data["final"]
            save()
            return
        COMM_LOG.append({"event": "msg", "buffers": [len(b) for b in msg.get("buffers", [])]})
        n = int(data.get("echo_bytes", 0))
        comm.send({"echo": n}, None, [bytes(n)])

    comm.on_msg(on_msg)


IPython.get_ipython().kernel.comm_manager.register_target("gigi_probe_comm", _on_comm_open)
output.register_callback("gigi_probe_callback", lambda *args, **kwargs: {"args": list(args), "kwargs": kwargs})
REPORT["comm"] = "no final message from the browser yet"
display(HTML(COMM_HTML))

## 输出 iframe 里有什么

`eval_js` 在**当前单元格**的输出 iframe 里执行一段 JS，并阻塞 Python 直到拿到返回值。

In [ ]:
FRAME_JS = r'''
// Runs inside the output frame of the current cell (via google.colab.output.eval_js).
// Reports what that frame can do. Identifiers in the origin are masked.
(async () => {
  const out = {};
  const mask = (s) => String(s).replace(/[a-z0-9]{10,}/gi, '<id>');
  out.origin = mask(location.origin);
  out.secure = self.isSecureContext;
  out.crossOriginIsolated = self.crossOriginIsolated;
  out.userAgent = navigator.userAgent;
  out.devicePixelRatio = self.devicePixelRatio;
  out.fullscreenEnabled = document.fullscreenEnabled;
  out.api = {
    BroadcastChannel: typeof BroadcastChannel,
    CompressionStream: typeof CompressionStream,
    OffscreenCanvas: typeof OffscreenCanvas,
    SharedArrayBuffer: typeof SharedArrayBuffer,
    clipboard: typeof navigator.clipboard,
  };
  try {
    out.framesInParent = parent.frames.length;
  } catch (e) { out.framesInParent = String(e); }
  try {
    out.allowedFeatures = document.featurePolicy ? document.featurePolicy.allowedFeatures().sort() : null;
  } catch (e) { out.allowedFeatures = String(e); }
  try {
    const c = (self.google || {}).colab || {};
    out.colabApi = {
      kernel: Object.keys(c.kernel || {}).sort(),
      output: Object.keys(c.output || {}).sort(),
      comms: Object.keys((c.kernel || {}).comms || {}).sort(),
      accessAllowed: (c.kernel || {}).accessAllowed,
    };
  } catch (e) { out.colabApi = String(e); }

  out.webgpu = { present: 'gpu' in navigator };
  if (navigator.gpu) {
    try {
      const adapter = await navigator.gpu.requestAdapter();
      if (!adapter) {
        out.webgpu.adapter = null;
      } else {
        const info = adapter.info || {};
        out.webgpu.adapter = {
          vendor: info.vendor, architecture: info.architecture,
          device: info.device, description: info.description,
        };
        out.webgpu.features = [...adapter.features].sort();
        const L = adapter.limits;
        out.webgpu.limits = {};
        for (const k of ['maxTextureDimension2D', 'maxBufferSize', 'maxStorageBufferBindingSize',
          'maxStorageBuffersPerShaderStage', 'maxStorageTexturesPerShaderStage',
          'maxComputeInvocationsPerWorkgroup', 'maxColorAttachments', 'maxBindGroups']) {
          out.webgpu.limits[k] = L[k];
        }
        out.webgpu.preferredCanvasFormat = navigator.gpu.getPreferredCanvasFormat();

        // A tiny compute job with readback, to prove the device really works in this frame.
        const t0 = performance.now();
        const device = await adapter.requestDevice();
        const module = device.createShaderModule({ code: `
          @group(0) @binding(0) var<storage, read_write> d: array<u32>;
          @compute @workgroup_size(4)
          fn main(@builtin(global_invocation_id) i: vec3<u32>) { d[i.x] = i.x * i.x + 1u; }` });
        const pipeline = device.createComputePipeline({ layout: 'auto', compute: { module, entryPoint: 'main' } });
        const storage = device.createBuffer({ size: 16, usage: GPUBufferUsage.STORAGE | GPUBufferUsage.COPY_SRC });
        const readback = device.createBuffer({ size: 16, usage: GPUBufferUsage.MAP_READ | GPUBufferUsage.COPY_DST });
        const bindGroup = device.createBindGroup({
          layout: pipeline.getBindGroupLayout(0),
          entries: [{ binding: 0, resource: { buffer: storage } }],
        });
        const encoder = device.createCommandEncoder();
        const pass = encoder.beginComputePass();
        pass.setPipeline(pipeline); pass.setBindGroup(0, bindGroup); pass.dispatchWorkgroups(1); pass.end();
        encoder.copyBufferToBuffer(storage, 0, readback, 0, 16);
        device.queue.submit([encoder.finish()]);
        await readback.mapAsync(GPUMapMode.READ);
        out.webgpu.compute = Array.from(new Uint32Array(readback.getMappedRange()));
        out.webgpu.computeExpected = [1, 2, 5, 10];
        out.webgpu.deviceAndComputeMs = Math.round(performance.now() - t0);
        device.destroy();
      }
    } catch (e) { out.webgpu.error = String(e); }
  }
  return out;
})()
'''

probe("frame", lambda: output.eval_js(FRAME_JS, timeout_sec=90))

## 跨单元格访问

这段 JS 运行在*本单元格*的 iframe 里，目标是上面 frame A 所在的*另一个* iframe。

In [ ]:
SIBLING_JS = r'''
// Runs in the output frame of a LATER cell. Tries to reach "frame A", which an
// earlier cell created, by two routes:
//   direct    - walk parent.frames and call a function on the sibling's window
//               (what Colab's own google.colab.html proxy does)
//   broadcast - BroadcastChannel, which only needs the frames to share an origin
(async () => {
  const out = { direct: null, broadcast: null };
  try {
    for (let i = 0; i < parent.frames.length; i++) {
      try {
        const w = parent.frames[i];
        if (w !== self && w.gigiProbeMarker === 'frame-A') {
          out.direct = w.gigiProbePing('direct');
          break;
        }
      } catch (e) { /* cross-origin frame: skip */ }
    }
  } catch (e) { out.directError = String(e); }
  try {
    const bc = new BroadcastChannel('gigi-probe');
    out.broadcast = await new Promise((resolve) => {
      const timer = setTimeout(() => resolve(null), 2000);
      bc.onmessage = (ev) => {
        if (ev.data && ev.data.pong) { clearTimeout(timer); resolve('pong:' + ev.data.pong); }
      };
      bc.postMessage({ ping: 'broadcast' });
    });
    bc.close();
  } catch (e) { out.broadcastError = String(e); }
  return out;
})()
'''

probe("sibling_frames", lambda: output.eval_js(SIBLING_JS, timeout_sec=30))

## `eval_js` 的延迟与载荷上限

Colab 自己的上传代码把浏览器 → 内核的数据切成 100 KB 一块，下载代码按 1 MB 一块。
这里直接量一下两个方向各能走多大。

In [ ]:
def eval_js_limits():
    r = {}
    times = []
    for _ in range(20):
        t = time.perf_counter()
        output.eval_js("1", timeout_sec=10)
        times.append((time.perf_counter() - t) * 1000)
    times.sort()
    r["roundtrip_ms"] = {"min": round(times[0], 1), "median": round(times[10], 1), "max": round(times[-1], 1)}

    for direction in ("kernel_to_browser", "browser_to_kernel"):
        r[direction] = []
        for n in (100_000, 1_000_000, 4_000_000, 16_000_000):
            # Saved first, so a dropped connection still leaves a trace of where it happened.
            REPORT["eval_js_in_progress"] = {"direction": direction, "bytes": n, "so_far": r}
            save()
            t = time.perf_counter()
            step = {"bytes": n}
            try:
                if direction == "kernel_to_browser":
                    got = output.eval_js('("' + "x" * n + '").length', timeout_sec=60)
                    step["ok"] = got == n
                else:
                    got = output.eval_js(f'"x".repeat({n})', timeout_sec=60)
                    step["ok"] = isinstance(got, str) and len(got) == n
            except Exception as e:
                step["ok"] = False
                step["error"] = f"{type(e).__name__}: {e}"[:200]
            step["seconds"] = round(time.perf_counter() - t, 2)
            r[direction].append(step)
            if not step["ok"]:
                break
    REPORT.pop("eval_js_in_progress", None)
    return r


probe("eval_js", eval_js_limits)

## 内核里的后台 HTTP 服务

服务跑在内核进程的一个后台线程里。浏览器通过 Colab 的端口代理访问它，
而此时主线程正卡在 `eval_js` 上等结果，也就是"单元格执行期间"。

In [ ]:
# A tiny HTTP server that runs in a background thread of the kernel process.
# Standard library only. The browser reaches it through Colab's kernel-port proxy,
# even while the main thread is busy executing a cell.
import base64
import hashlib
import http.server
import json
import socket
import socketserver
import threading
import time

HTTP_LOG = []  # what the kernel actually received


class ProbeHandler(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        pass

    def _send(self, code, body=b"", ctype="application/octet-stream"):
        origin = self.headers.get("Origin")
        self.send_response(code)
        self.send_header("Access-Control-Allow-Origin", origin or "*")
        if origin:
            self.send_header("Access-Control-Allow-Credentials", "true")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "content-type")
        self.send_header("Vary", "Origin")
        self.send_header("Cache-Control", "no-store")
        # Ask Colab not to copy these responses into the notebook file.
        self.send_header("x-colab-notebook-cache-control", "no-cache")
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_OPTIONS(self):
        self._send(204)

    def do_GET(self):
        path = self.path.split("?")[0]
        HTTP_LOG.append({"method": "GET", "path": path})
        if path.endswith("/ws"):
            return self._websocket_echo_once()
        if path.endswith("/bytes"):
            n = int(self.path.split("n=")[1])
            return self._send(200, bytes(n))
        body = json.dumps({"ok": True, "kernel_time": time.time()}).encode()
        self._send(200, body, "application/json")

    def do_POST(self):
        n = int(self.headers.get("Content-Length", 0))
        body = self.rfile.read(n)
        HTTP_LOG.append({"method": "POST", "path": self.path, "bytes": len(body)})
        self._send(200, json.dumps({"received": len(body)}).encode(), "application/json")

    def _websocket_echo_once(self):
        # Minimal RFC 6455 handshake, then echo one short text frame.
        key = self.headers.get("Sec-WebSocket-Key")
        if not key:
            return self._send(400, b"not a websocket request")
        magic = "258EAFA5-E914-47DA-95CA-C5AB0DC85B11"
        accept = base64.b64encode(hashlib.sha1((key + magic).encode()).digest()).decode()
        self.send_response(101)
        self.send_header("Upgrade", "websocket")
        self.send_header("Connection", "Upgrade")
        self.send_header("Sec-WebSocket-Accept", accept)
        self.end_headers()
        head = self.rfile.read(2)
        length = head[1] & 0x7F  # client frames are masked; payload here is < 126 bytes
        mask = self.rfile.read(4)
        payload = bytes(b ^ mask[i % 4] for i, b in enumerate(self.rfile.read(length)))
        HTTP_LOG.append({"method": "WS", "payload": payload.decode(errors="replace")})
        self.wfile.write(bytes([0x81, len(payload)]) + payload)
        self.wfile.flush()
        self.close_connection = True


class _DualStackServer(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True
    allow_reuse_address = True
    address_family = socket.AF_INET6

    def server_bind(self):
        # Accept IPv4 and IPv6, whichever "localhost" resolves to for the proxy.
        self.socket.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 0)
        super().server_bind()


class _IPv4Server(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True
    allow_reuse_address = True


def start_probe_server():
    """Start the server on a free port; returns (server, port)."""
    try:
        server = _DualStackServer(("::", 0), ProbeHandler)
    except OSError:
        server = _IPv4Server(("", 0), ProbeHandler)
    threading.Thread(target=server.serve_forever, daemon=True, name="gigi-probe-http").start()
    return server, server.server_address[1]


HTTP_JS = r'''
// Runs in an output frame WHILE the Python cell is still executing.
// Talks to a small HTTP server that lives in a background thread of the kernel,
// through Colab's kernel-port proxy. The port number is substituted on the last line.
(async (port) => {
  const out = {};
  const mask = (s) => String(s).replace(/[a-z0-9]{10,}/gi, '<id>');
  const timed = async (label, fn) => {
    const t0 = performance.now();
    try {
      const extra = await fn();
      out[label] = Object.assign({ ok: true, ms: Math.round(performance.now() - t0) }, extra || {});
    } catch (e) {
      out[label] = { ok: false, ms: Math.round(performance.now() - t0), error: String(e).slice(0, 200) };
    }
  };
  let base = null;
  await timed('proxyPort', async () => {
    base = await google.colab.kernel.proxyPort(port, { cache: false });
    return { urlPattern: mask(base) };
  });
  if (base) {
    const u = (p) => new URL(p, base).toString();
    await timed('getSmall', async () => {
      const r = await fetch(u('small'));
      return { status: r.status, body: (await r.text()).slice(0, 80) };
    });
    await timed('getSmallWithCredentials', async () => {
      const r = await fetch(u('small'), { credentials: 'include' });
      return { status: r.status };
    });
    await timed('get8MB', async () => {
      const r = await fetch(u('bytes?n=8000000'));
      return { status: r.status, bytes: (await r.arrayBuffer()).byteLength };
    });
    await timed('post8MB', async () => {
      const r = await fetch(u('upload'), { method: 'POST', body: new Blob([new Uint8Array(8000000)]) });
      return { status: r.status, body: (await r.text()).slice(0, 80) };
    });
    await timed('webSocket', () => new Promise((resolve, reject) => {
      const ws = new WebSocket(u('ws').replace(/^http/, 'ws'));
      const timer = setTimeout(() => { try { ws.close(); } catch (e) {} reject(new Error('timeout')); }, 8000);
      ws.onopen = () => ws.send('ping');
      ws.onmessage = (ev) => { clearTimeout(timer); ws.close(); resolve({ echo: String(ev.data) }); };
      ws.onerror = () => { clearTimeout(timer); reject(new Error('websocket error')); };
    }));
  }
  // The route google.colab.html uses: the Colab service worker maps https://localhost:<port>.
  await timed('serviceWorkerLocalhost', async () => {
    const r = await fetch('https://localhost:' + port + '/small');
    return { status: r.status, body: (await r.text()).slice(0, 80) };
  });
  return out;
})(__PORT__)
'''

_server, PORT = start_probe_server()


def kernel_http():
    browser = output.eval_js(HTTP_JS.replace("__PORT__", str(PORT)), timeout_sec=180)
    return {"port": PORT, "browser": browser, "kernel_saw": HTTP_LOG[:20]}


probe("kernel_http", kernel_http)

## 内核里的无头 WebGPU：Node + Dawn

Gigi 的 WebGPU 单元测试就是这样跑的（`npm install webgpu` 后 `node .`）。
没有 GPU 的 VM 需要一个软件 Vulkan 驱动；如果第一次找不到 adapter，就装 Mesa 的 lavapipe 再试。

In [ ]:
NODE_PROBE = r'''
// Runs in the Colab VM under Node, using the Dawn-based "webgpu" npm package:
// the same headless route Gigi's own WebGPU unit tests use ("node .").
import { create, globals } from 'webgpu';
Object.assign(globalThis, globals);

const out = {};
try {
  const gpu = create([]);
  const adapter = await gpu.requestAdapter();
  if (!adapter) {
    out.adapter = null;
  } else {
    const info = adapter.info || {};
    out.adapter = {
      vendor: info.vendor, architecture: info.architecture,
      device: info.device, description: info.description,
    };
    out.features = [...adapter.features].sort();
    const device = await adapter.requestDevice();
    device.pushErrorScope('validation');

    // 1024x1024 compute job writing a storage buffer, then readback.
    const N = 1024 * 1024;
    const t0 = performance.now();
    const module = device.createShaderModule({ code: `
      @group(0) @binding(0) var<storage, read_write> d: array<u32>;
      @compute @workgroup_size(64)
      fn main(@builtin(global_invocation_id) i: vec3<u32>) { d[i.x] = i.x * 3u + 1u; }` });
    const pipeline = device.createComputePipeline({ layout: 'auto', compute: { module, entryPoint: 'main' } });
    const storage = device.createBuffer({ size: N * 4, usage: GPUBufferUsage.STORAGE | GPUBufferUsage.COPY_SRC });
    const readback = device.createBuffer({ size: N * 4, usage: GPUBufferUsage.MAP_READ | GPUBufferUsage.COPY_DST });
    const bindGroup = device.createBindGroup({
      layout: pipeline.getBindGroupLayout(0),
      entries: [{ binding: 0, resource: { buffer: storage } }],
    });
    const encoder = device.createCommandEncoder();
    const pass = encoder.beginComputePass();
    pass.setPipeline(pipeline); pass.setBindGroup(0, bindGroup); pass.dispatchWorkgroups(N / 64); pass.end();
    encoder.copyBufferToBuffer(storage, 0, readback, 0, N * 4);
    device.queue.submit([encoder.finish()]);
    await readback.mapAsync(GPUMapMode.READ);
    const data = new Uint32Array(readback.getMappedRange());
    out.computeOk = data[0] === 1 && data[5] === 16 && data[N - 1] === (N - 1) * 3 + 1;
    out.computeMs = Math.round(performance.now() - t0);
    const error = await device.popErrorScope();
    out.validationError = error ? error.message : null;
  }
} catch (e) {
  out.error = String(e);
}
console.log('RESULT ' + JSON.stringify(out));
process.exit(0);
'''


def node_dawn():
    r = {"node": sh("node --version"), "npm": sh("npm --version")}
    if not r["node"].startswith("v"):
        r["note"] = "no Node.js in this VM"
        return r
    d = os.path.join(WORKDIR, "gigi_probe_node")
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d, "probe.mjs"), "w") as f:
        f.write(NODE_PROBE)
    t = time.time()
    r["npm_install"] = sh(f"cd {d} && (npm init -y >/dev/null 2>&1; npm install webgpu 2>&1 | tail -3)", timeout=600)
    r["npm_install_seconds"] = round(time.time() - t, 1)
    r["package_mb"] = sh(f"du -sm {d}/node_modules/webgpu | cut -f1")

    def run():
        text = sh(f"cd {d} && node probe.mjs 2>&1 | tail -12", timeout=180, limit=20000)
        for line in text.splitlines():
            if line.startswith("RESULT "):
                return json.loads(line[len("RESULT "):])
        return {"raw": text[-1500:]}

    r["first_run"] = run()
    if not r["first_run"].get("adapter"):
        t = time.time()
        install = "apt-get install -y -qq mesa-vulkan-drivers"
        r["apt_install"] = sh(f"({install} || (apt-get update -qq && {install})) 2>&1 | tail -3", timeout=900)
        r["apt_install_seconds"] = round(time.time() - t, 1)
        r["vulkan_icd_files"] = sorted(glob.glob("/usr/share/vulkan/icd.d/*"))
        r["after_lavapipe"] = run()
    return r


probe("node_dawn", node_dawn)

## 持久化：保存后重开，画面还活着吗

下面这个输出完全自包含，不依赖内核。运行后应当看到一块会动的彩色画面。

**人工检查**（这一步脚本测不了）：

1. 全部运行结束后保存（Ctrl/Cmd + S），等标题栏显示已保存。
2. 刷新页面，**不要**连接运行时。
3. 滚动到这个单元格：画面还在动吗？下面那行字里 `kernel access` 显示什么？

把观察到的结果和 JSON 一起告诉我即可。

In [ ]:
PERSIST_HTML = r'''
<div style="font: 13px monospace; line-height: 1.5">
  <canvas id="gigi-probe-canvas" width="256" height="256" style="border: 1px solid #888"></canvas>
  <div id="gigi-probe-persist">persistence probe: starting…</div>
</div>
<script type="module">
// Self-contained: nothing here needs the kernel. If Colab re-runs saved outputs when a
// notebook is opened, this canvas animates again without any runtime attached.
const label = document.getElementById('gigi-probe-persist');
const emittedAt = __EMITTED_AT__;  // kernel clock, seconds, baked in when the cell ran
const kernel = ((self.google || {}).colab || {}).kernel;
const access = kernel ? kernel.accessAllowed : 'no google.colab';
const age = Math.round(Date.now() / 1000 - emittedAt);
const status = (gpu) => {
  label.textContent = `persistence probe | output is ${age}s old | kernel access: ${access} | WebGPU: ${gpu}`;
};
status('starting');
try {
  if (!navigator.gpu) throw new Error('navigator.gpu missing');
  const adapter = await navigator.gpu.requestAdapter();
  if (!adapter) throw new Error('no adapter');
  const device = await adapter.requestDevice();
  const canvas = document.getElementById('gigi-probe-canvas');
  const context = canvas.getContext('webgpu');
  const format = navigator.gpu.getPreferredCanvasFormat();
  context.configure({ device, format });
  const module = device.createShaderModule({ code: `
    @group(0) @binding(0) var<uniform> time: f32;
    @vertex fn vs(@builtin(vertex_index) i: u32) -> @builtin(position) vec4f {
      var p = array<vec2f, 3>(vec2f(-1, -1), vec2f(3, -1), vec2f(-1, 3));
      return vec4f(p[i], 0, 1);
    }
    @fragment fn fs(@builtin(position) p: vec4f) -> @location(0) vec4f {
      let uv = p.xy / 256.0;
      return vec4f(0.5 + 0.5 * cos(time + uv.xyx * 3.0 + vec3f(0, 2, 4)), 1);
    }` });
  const pipeline = device.createRenderPipeline({
    layout: 'auto',
    vertex: { module, entryPoint: 'vs' },
    fragment: { module, entryPoint: 'fs', targets: [{ format }] },
  });
  const uniform = device.createBuffer({ size: 16, usage: GPUBufferUsage.UNIFORM | GPUBufferUsage.COPY_DST });
  const bindGroup = device.createBindGroup({
    layout: pipeline.getBindGroupLayout(0),
    entries: [{ binding: 0, resource: { buffer: uniform } }],
  });
  let frames = 0;
  const frame = (ms) => {
    device.queue.writeBuffer(uniform, 0, new Float32Array([ms / 1000, 0, 0, 0]));
    const encoder = device.createCommandEncoder();
    const pass = encoder.beginRenderPass({ colorAttachments: [{
      view: context.getCurrentTexture().createView(), loadOp: 'clear', storeOp: 'store',
    }] });
    pass.setPipeline(pipeline); pass.setBindGroup(0, bindGroup); pass.draw(3); pass.end();
    device.queue.submit([encoder.finish()]);
    frames++;
    if (frames % 30 === 1) status(`animating (${frames} frames)`);
    requestAnimationFrame(frame);
  };
  self.gigiProbeFrames = () => frames;
  requestAnimationFrame(frame);
} catch (e) {
  status('FAILED: ' + e.message);
}
</script>
'''

display(HTML(PERSIST_HTML.replace("__EMITTED_AT__", str(int(time.time())))))

## 报告

如果 `comm` 一项还显示 "no final message"，等几秒后单独重跑这个单元格
（comm 消息只有在内核空闲时才会被处理）。

In [ ]:
REPORT["comm_kernel_log"] = COMM_LOG
save()
print(json.dumps(REPORT, indent=1, default=str))